In [1]:
def main(datasource="bigalpha_2026_stock_bar1m", start_date=None, end_date=None):
    import pandas as pd
    import dai

    table_name = datasource or "bigalpha_2026_stock_bar1m"
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    query_start = (start_ts - pd.Timedelta(days=45)).strftime("%Y-%m-%d 00:00:00")
    query_end = end_ts.strftime("%Y-%m-%d 23:59:59")

    sql = f"""
    WITH raw_bar AS (
        SELECT
            date,
            date::DATE::DATETIME AS obs_date,
            instrument,
            close AS price,
            volume,
            amount,
            CASE
                WHEN ask_price1 > 0 AND bid_price1 > 0
                THEN (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0)
                ELSE NULL
            END AS relative_spread,
            date_part('hour', date) AS hh,
            date_part('minute', date) AS mm
        FROM {table_name}
        WHERE close > 0 AND volume > 0 AND amount >= 0
    ),
    minute_base AS (
        SELECT
            date,
            obs_date,
            instrument,
            price,
            volume,
            amount,
            relative_spread,
            CASE
                WHEN (hh = 9 AND mm >= 31)
                    OR hh = 10
                    OR (hh = 11 AND mm <= 30)
                THEN hh * 60 + mm - 571
                WHEN (hh = 13 AND mm >= 1)
                    OR (hh = 14 AND mm <= 57)
                THEN 120 + hh * 60 + mm - 781
                ELSE NULL
            END AS slot
        FROM raw_bar
    ),
    daily_all AS (
        SELECT
            obs_date AS date,
            instrument,
            COUNT(*) AS valid_count,
            SUM(amount) AS total_amount,
            median(relative_spread) AS median_relative_spread
        FROM minute_base
        WHERE slot BETWEEN 0 AND 236
        GROUP BY obs_date, instrument
    ),
    daily_late AS (
        SELECT
            obs_date AS date,
            instrument,
            COUNT(*) AS late_valid_count,
            SUM(amount) AS late_amount,
            arg_min(price, slot) AS late_start_price,
            arg_max(price, slot) AS late_end_price
        FROM minute_base
        WHERE slot BETWEEN 200 AND 236
        GROUP BY obs_date, instrument
    ),
    daily_factor AS (
        SELECT
            a.date,
            a.instrument,
            CASE
                WHEN a.valid_count >= 80
                    AND l.late_valid_count >= 20
                    AND a.total_amount > 0
                    AND l.late_amount >= 0
                    AND l.late_start_price > 0
                    AND l.late_end_price > 0
                THEN
                    (l.late_amount / (a.total_amount + 1e-12))
                    * exp(-200.0 * abs(log(l.late_end_price / l.late_start_price)))
                    * log(1.0 + a.total_amount / 100000000.0)
                    / (1.0 + 50.0 * coalesce(a.median_relative_spread, 0.0))
                ELSE NULL
            END AS tail_passive_rebalance_absorption_daily
        FROM daily_all a
        LEFT JOIN daily_late l
        ON a.date = l.date AND a.instrument = l.instrument
    ),
    rolling_factor AS (
        SELECT
            date,
            instrument,
            CASE
                WHEN COUNT(tail_passive_rebalance_absorption_daily) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
                ) >= 12
                THEN AVG(tail_passive_rebalance_absorption_daily) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
                )
                ELSE NULL
            END AS tail_passive_rebalance_absorption_20d
        FROM daily_factor
    )
    SELECT
        date,
        instrument,
        -tail_passive_rebalance_absorption_20d AS factor
    FROM rolling_factor
    WHERE tail_passive_rebalance_absorption_20d IS NOT NULL
    ORDER BY date, instrument
    """

    df = dai.query(
        sql,
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start_ts.normalize()) & (df["date"] <= end_ts.normalize())]
    return df[["date", "instrument", "factor"]]
